<a href="https://colab.research.google.com/github/matkins8943/matkins8943.github.io/blob/main/Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **UFO Sighting Analysis**

**By:** Macey Atkins and Saanvi Nair

**Project Site:** https://matkins8943.github.io/

# **Project Goals**
We are considering working on datasets related to paranormal UFO sightings/encounters.
The main question we are hoping to answer is whether or not we can predict the time and/or place of future UFO sightings based on previous data statistics and datasets. Our hypothesis is that most current sightings will be along the coastlines or borders with another country, and will most likely take place at night.
We have currently found two datasets related to UFO sightings and object reports. We plan to use this data to aid our continued research.

# **Collaboration Plan**
We plan to continue using text messaging as the primary form of communication. We have created a private Github repository and plan to work on the project via phone calls and in person meetings once a week. Specific meeting dates will be dependent upon weekly availability.

# **1. Setup**
This is where we handle and add all the imports.

In [15]:
#Dataset 1 imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import io
import plotly.express as px

#Dataset 2 imports
!pip install geonamescache -q
import geonamescache

#Graph 3 imports
import plotly.graph_objects as go

# **2. Loading and Tidying the Datasets**
In this section we will load and tidy datasets 1 and 2.

# **2.1 Dataset 1: Loading and Tidying**

Our first dataset was derived from [Kaggle](https://www.kaggle.com/datasets/ahemateja19bec1025/ufosightingsdataset). It measures UFO sightings across the years with the data, name, city/state/country, and description of the sighting. This set allows us to visualize the number of sightings over the United States per state. We want to know where most sightings occur and whether or not the region of the U.S. plays a role in reported sightings.


In [16]:
#reads the file, replaces "None" values with NaN
df1 = pd.read_csv('https://raw.githubusercontent.com/matkins8943/matkins8943.github.io/refs/heads/main/UFO_Sightings.csv')
df1 = df1.replace("None", np.nan)

#data types
print(df1.dtypes)

#creates the a dataframe of just observations (sightings) that occurred in the U.S.
unitedStates = df1[df1["Country"] == "United States"].copy()
#split "City, State" into 2 columns "City" and "State" to improve tidy data
unitedStates[['City', 'State']] = (unitedStates['City, State'].str.split(',', n=1, expand=True))

#removes irrelevant columns from the dataframe
unitedStates = unitedStates[['Date', 'Name', 'City', 'State', 'Country', 'Description']].copy()
#if only the "State" is listed (no city), then fill the "State" column with whatever is in the "City" column
unitedStates["State"] = unitedStates["State"].fillna(unitedStates["City"])
display(unitedStates.head())

Unnamed: 0      int64
Date           object
Name           object
City, State    object
Country        object
Description    object
Unnamed: 6     object
dtype: object


,Date,Name,City,State,Country,Description
11,1896–1897,Mystery airships,NaN,NaN,United States,Numerous reports of UFO sightings and attempte...
12,1897-04-17,"Aurora, Texas, UFO incident",Aurora,Texas,United States,A tale of a UFO crash and a burial of its alie...
16,1941,Cape Girardeau UFO crash,Cape Girardeau,Missouri,United States,First responders and a Baptist minister allege...
17,24-02-1942,Battle of Los Angeles,Los Angeles,California,United States,Unidentified aerial objects trigger the firing...
20,21-06-1947,Maury Island incident,Puget Sound near Maury Island,Washington,United States,Harold A. Dahl reported that his dog was kille...


# **2.2 Dataset 2: Loading and Tidying**

We found our second dataset from a [public repository](https://github.com/syamkakarla98/DataScience_Head_Start/blob/master/DataAnalysis/UFO_Sightings.ipynb) on GitHub, free for public use. It utilizes civilian reporting and was created by the National UFO reporting center. It also measures UFO sightings across the United States, but with higher specificity, listing the shapes that were identified from the sightings. We were curious about whether the date and time of the sighting played a role in the shape that was observed.

In [17]:
#reads the file and replaces "None" text with NaN (missing values)
df2 = pd.read_csv('https://bit.ly/uforeports')
df2 = df2.replace("None", np.nan)

#data types
print(df2.dtypes)

#split the date and time into separate columns (tidy data: one variable per column)
df2[['date', 'time']] = df2['Time'].str.split(' ', n=1, expand=True)
#drop the original "Time" column because it still contains both values
df2 = df2.drop(columns=['Time'])

display(df2.head())

City               object
Colors Reported    object
Shape Reported     object
State              object
Time               object
dtype: object


,City,Colors Reported,Shape Reported,State,date,time
0,Ithaca,NaN,TRIANGLE,NY,6/1/1930,22:00
1,Willingboro,NaN,OTHER,NJ,6/30/1930,20:00
2,Holyoke,NaN,OVAL,CO,2/15/1931,14:00
3,Abilene,NaN,DISK,KS,6/1/1931,13:00
4,New York Worlds Fair,NaN,LIGHT,NY,4/18/1933,19:00


# **2.3 Dataset 3: Loading and Tidying**

Our third dataset was derived through a combination of dataset 1 and 2 features.

In [18]:
#convert states to their corresponding abbreviations
stateabb = ({
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT",
    "Delaware": "DE", "Florida": "FL", "Georgia": "GA", "Hawaii": "HI",
    "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA",
    "Kansas": "KS", "Kentucky": "KY", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN",
    "Mississippi": "MS", "Missouri": "MO", "Montana": "MT", "Nebraska": "NE",
    "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM",
    "New York": "NY", "North Carolina": "NC", "North Dakota": "ND",
    "Ohio": "OH", "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA",
    "Rhode Island": "RI", "South Carolina": "SC", "South Dakota": "SD",
    "Tennessee": "TN", "Texas": "TX", "Utah": "UT", "Vermont": "VT",
    "Washington": "WA", "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY",
    "Virginia": "VA"
})

#Dataset 3: hand-built table of coastal and border states
coastal_states = ['AL','AK','CA','CT','DE','FL','GA','HI','LA','ME','MD','MA','MS','NH',
                  'NJ','NY','NC','OR','RI','SC','TX','VA','WA']
border_states  = ['AK','AZ','CA','ID','ME','MI','MN','MT','NH','NM','NY','ND','OH','PA',
                  'TX','VT','WA','WI']

In [19]:
#tidy format: one row per state, one column per variable
state_info = pd.DataFrame({'State': list(stateabb.values())})
state_info['coastal'] = state_info['State'].isin(coastal_states)
state_info['border']  = state_info['State'].isin(border_states)

#label each state by its region type (used by the map and the group stats)
def region_type(row):
    if row['coastal'] and row['border']: return 'Coastal and border'
    if row['coastal']:                   return 'Coastal only'
    if row['border']:                    return 'Border only'
    return 'Neither'

state_info['Region type'] = state_info.apply(region_type, axis=1)

#merge with Dataset 2: sightings per state (states with no sightings get 0)
state_counts = t['State'].value_counts().rename_axis('State').reset_index(name='Sightings')
state_info = state_info.merge(state_counts, on='State', how='left')
state_info['Sightings'] = state_info['Sightings'].fillna(0)

#check that every state code in Dataset 2 matched a state in our table
unmatched = set(t['State']) - set(state_info['State'])
print("State codes in Dataset 2 not in our table:", unmatched)

display(state_info.head())
print(state_info['Region type'].value_counts())

NameError: name 't' is not defined

# **3. EDA: Summary Statistics and Tables**
Now that the data has been loaded and tidied, it is ready to be visualized. These graphs convey summary statistics and a more generalized analysis of the data.

# 3.1 Dataset 1: Summary Stats

In [ ]:
#some "States" are listed with other words (ex: "mostly Washington"), in order
#to decipher the state within, we use a for loop function to search if a
#state is in the "State" string
def find_state(key):
  if pd.isna(key):
    return np.nan
  for state, abbreviation in stateabb.items():
    if state in key:
      return abbreviation
  return np.nan

#strip states and replace them with their abbreviations
states = unitedStates['State'].str.strip()
newstateabb = states.apply(find_state)

#number of sightings (how many times each state has a reported sighting)
sightings = (newstateabb.value_counts().rename_axis('State').reset_index(name='Sightings'))

#Summary stats of the data
print(unitedStates.describe())
print(f"Average per state:   {sightings['Sightings'].mean():,.0f}")
print(f"Median per state:    {sightings['Sightings'].median():,.0f}")
#top and bottom 10 states
print("Top 10 states")
print(sightings.head(10))
print("Bottom 10 states")
print(sightings.tail(10))
#replaced with NaN
print("Number of missing values:")
print(unitedStates.isna().sum())

Based off the summary stats, we can see that the state with the most UFO findings are Texas and New Mexico, both having 4 observations. Regarding the states with the lowest number of observations, many only have 1, so it is difficult to rank them. The average number of observations per state is 2, but the median is 1, so if plotted as a bar graph, it would be right skewed. There are 2 missing values for both cities and states that have been replaced with NaN.

# 3.2 Dataset 2: Shape Counts and Peak Hours

In [ ]:
#total sightings and data range
print("Total sightings: ", len(t))
print("Date range: ", t["dt"].min(), "to", t["dt"].max())

#Shape breakdown
shape_counts = t["shape_group"].value_counts()
shape_summary = pd.DataFrame({
    "count of shapes": shape_counts,
    "percent of total findings": (shape_counts / len(t) * 100).round(1)
})

# Peak hour of observation for each shape, formatted as AM/PM
def format_hour(h):
    return pd.Timestamp(2000, 1, 1, h).strftime('%I%p').lstrip('0')   #0 -> 12AM, 12 -> 12PM, 21 -> 9PM

peak_hour = t.groupby("shape_group")["hour_int"].agg(lambda s: s.mode().iloc[0])
shape_summary["peak hour"] = peak_hour.apply(format_hour)
display(shape_summary)

#replaced with NaN
print("Number of missing values:")
print(t.isna().sum())

From the summary stats we generated, we can see that the most common shape seen is light, making up 27.9% of observations, and the peak time that it is observed is at 9PM. Generally speaking, the peak time the top 7 symbols are observed are also around 9PM to 10PM. There are a total of 7494 sightings starting in year 1990 up to 2000 in this dataset. As far as unreported data, there are 8 cities and 5985 colors that are unreported, so only 1509 sightings had colors reported for them.

# 3.3 Dataset 3: Group Stats (Coastal and Border)

In [ ]:
for col in ['coastal', 'border']:
    print(f"\n{col.capitalize()} states vs. others")
    display(state_info.groupby(col)['Sightings'].agg(['count', 'mean', 'median', 'std']).round(1))

display(state_info.groupby('Region type')['Sightings'].agg(['count', 'mean', 'median', 'sum']).round(1))

These group statistics show us that costal states and border states tend to have higher numbers of reported sightings. States that are both costal and border have exponentially higher numbers. From this data, we are able to see that sightings are more likley to occur in states such as Texas, Washington, Alaska, California, New York, New Hampshire, and Maine.

# 3.4 Cross-Tab of Shape × Time of Day

In [ ]:
def time_bucket(h):
    if h >= 20 or h < 5:  return 'Night (8PM-4AM)'
    if h < 12:            return 'Morning (5AM-11AM)'
    if h < 17:            return 'Afternoon (12PM-4PM)'
    return 'Evening (5PM-7PM)'

t['time_of_day'] = t['hour_int'].apply(time_bucket)

order = ['Morning (5AM-11AM)', 'Afternoon (12PM-4PM)', 'Evening (5PM-7PM)', 'Night (8PM-4AM)']

#counts
display(pd.crosstab(t['shape_group'], t['time_of_day'])[order])

In [ ]:
#percent of each shape's sightings in each time bucket (each row adds to 100)
ct = pd.crosstab(t['shape_group'], t['time_of_day'], normalize='index')[order] * 100
display(ct.round(1))

# **4. EDA: Where?**

# 4.1 Graph 1: Where Do Sightings Occur Across the Map?
We created a heatmap to visualize the spatial distribution of the data. States with more reported sightings are represented in darker shades of blue, while states with fewer sightings are lighter.

In [ ]:
#creates the U.S. map
fig = px.choropleth(
    sightings,
    locations='State',
    locationmode='USA-states',
    color='Sightings',
    hover_name="State",
    color_continuous_scale='Blues',
    scope='usa',
    title="Sightings per U.S. State"
)

#layout of U.S. map figure
fig.update_layout(
    geo=dict(showframe=False, showcoastlines=True, projection_type='albers usa'),
    margin=dict(l=0, r=0, t=50, b=0)
)

fig.show()

As we can see, Texas and New Mexico have the highest number of reported sightings. The West Coast and Southern Boarder seem to have the most consistent set of sightings.

# 4.2 Graph 2: Coastal and Border States vs. Other States
We wanted to analyze the potential impact that location could have on the number of sightings, more specifically, the proximity of water bodies. This directly led us to analyzing the coastal and border states vs. other states.

In [ ]:
def region_type(row):
    if row['coastal'] and row['border']: return 'Coastal and border'
    if row['coastal']:                   return 'Coastal only'
    if row['border']:                    return 'Border only'
    return 'Neither'

state_info['Region type'] = state_info.apply(region_type, axis=1)

# base map: states shaded by number of sightings
fig = px.choropleth(
    state_info,
    locations='State',
    locationmode='USA-states',
    color='Sightings',
    hover_name='State',
    hover_data={'Region type': True, 'Sightings': True, 'State': False},
    color_continuous_scale='Blues',
    scope='usa',
    title='UFO Sightings per State (shaded) with Coastal and Border States (outlined)'
)

# outlines: one colored border per region type
outline_colors = {'Coastal and border': '#000000', 'Coastal only': '#e8710a', 'Border only': '#7b2cbf'}
for region, color in outline_colors.items():
    subset = state_info[state_info['Region type'] == region]
    fig.add_trace(go.Choropleth(
        locations=subset['State'], locationmode='USA-states',
        z=[1] * len(subset),
        colorscale=[[0, 'rgba(0,0,0,0)'], [1, 'rgba(0,0,0,0)']],
        showscale=False, hoverinfo='skip',
        marker_line_color=color, marker_line_width=3,
        showlegend=False
    ))
    # invisible point so the region shows up in the legend
    fig.add_trace(go.Scattergeo(
        lon=[None], lat=[None], mode='markers', name=region,
        marker=dict(symbol='square', size=12, color='white', line=dict(color=color, width=3))
    ))

fig.update_layout(
    geo=dict(showframe=False, projection_type='albers usa'),
    margin=dict(l=0, r=0, t=50, b=0),
    legend=dict(title='Region type (Neither = no outline)')
)
fig.show()

This heatmap gives us a much clearer visual as to which states are costal only, border only, or both coastal and border. It also allows us to combine our previous findings into one visual for better interpretation. The map shows us that the costal and border (black outline) states tend to have higher sighting numbers.

# **5. EDA: When?**

# 5.1 Graph 3: Sightings Per Year
We were curious whether seasonal differences played a role in the number of sightings observed. An indirect way of testing this observation was by graphing sightings by the month of observation, and seeing if they were clustered during a specific time of the year.

In [ ]:
# hand-built tidy table: one row per state, one column per variable
coastal_states = ['AL','AK','CA','CT','DE','FL','GA','HI','LA','ME','MD','MA','MS','NH',
                  'NJ','NY','NC','OR','RI','SC','TX','VA','WA']
border_states  = ['AK','AZ','CA','ID','ME','MI','MN','MT','NH','NM','NY','ND','OH','PA',
                  'TX','VT','WA','WI']
all_states = list(stateabb.values())

state_info = pd.DataFrame({'State': all_states})
state_info['coastal'] = state_info['State'].isin(coastal_states)
state_info['border']  = state_info['State'].isin(border_states)

# sightings per state from Dataset 2, joined to the flags (states with no sightings get 0)
state_counts = t['State'].value_counts().rename_axis('State').reset_index(name='Sightings')
state_info = state_info.merge(state_counts, on='State', how='left')
state_info['Sightings'] = state_info['Sightings'].fillna(0)

display(state_info.head())

# 5.2 Graph 4: Scatter of Hour vs. Date
Instead of organizing by location, we organized by the date and time of the sighting to see if there were correlations between the time of the sighting and the shape observed.

In [ ]:

import plotly.graph_objects as go

#convert time to a real object, removing rows that do not have a shape or date
df2["dt"] = pd.to_datetime(df2["date"].astype(str) + " " + df2["time"].astype(str),
                           format="%m/%d/%Y %H:%M", errors="coerce")

t = df2.dropna(subset=['dt', 'Shape Reported']).copy()
#converting the time of day to a number for the y-axis
t["hour"] = t["dt"].dt.hour + t["dt"].dt.minute / 60

#keeps 7 most common shapes
top = t["Shape Reported"].value_counts().head(7).index
t["shape_group"] = t["Shape Reported"].where(t["Shape Reported"].isin(top), "Other")

#doesn't include early years because of scarce data and keeps top 7 shapes
top = t["Shape Reported"].value_counts().head(7).index
t = t[t["Shape Reported"].isin(top) & (t["dt"].dt.year >= 1990)]

#making plot, with year on x axis and date on y axis, shapes plotted
fig = px.scatter(t, x="dt", y="hour", color="shape_group",
                 color_discrete_sequence=px.colors.qualitative.Vivid)
display(fig)


By looking at the plot, we can see that most of the sightings are clustered at the times before 5am, or after 8pm, indicating that they have a higher frequency at night. We can also see that they are clustered on the right side of the plot, indicating that observations have increased over time.

# 5.3 Graph 5: Sightings by Hour of Day
We categorized our data using the timestamps to determine whether there were times during the day where the number of sightings were the highest or lowest.

In [ ]:
fig = px.histogram(t, x='hour_int', nbins=24,
                   labels={'hour_int': 'Hour of day (0-23)'},
                   title='UFO Sightings by Hour of Day (1990-2000)')
fig.update_layout(yaxis_title='Number of sightings', bargap=0.05)
fig.show()

# share of sightings that happen at night (8 PM to 5 AM)
night = ((t['hour_int'] >= 20) | (t['hour_int'] < 5)).mean() * 100
print(f"Share of sightings between 8 PM and 5 AM: {night:.1f}%")

The sightings by hour of day tend to skew more to the left. Hours of the night appear to be when most sightings occur (along with a good cluster occuring in the early morning hours). This makes sense with our hypothesis since sightings are mainly easier to see in the dark.

In [ ]:
# Extract the year from the datetime column 'dt'
t['year'] = t['dt'].dt.year

# Group by 'year' using the correct DataFrame 't'
per_year = t.groupby('year').size().reset_index(name='Sightings')

fig = px.line(per_year, x='year', y='Sightings', markers=True,
              title='UFO Sightings per Year (1990-2000)')
fig.update_layout(xaxis_title='Year', yaxis_title='Number of sightings')
fig.show()
print(per_year)

Similarly to the hour of day, the sightings per year graph also tends to skew left. The number of sightings tends to rise with time, with 1999 holding the peak. This is likely correlated to the media/cultural events and technological advancements of this era.

# **6. EDA: Shape by Time-of-Day Heatmap**

# 6.1 Graph 6: Shape Seen by Time of the Day
Since we already graphed where all the different shapes were seen, we wanted to look at it from the lens of time. We wanted to analyze whther specific times favored sightings of specific shapes, and if there was any correlation to be observed.

In [ ]:
def time_bucket(h):
    if h >= 20 or h < 5:  return 'Night (8PM-4AM)'
    if h < 12:            return 'Morning (5AM-11AM)'
    if h < 17:            return 'Afternoon (12PM-4PM)'
    return 'Evening (5PM-7PM)'

t['hour_int'] = t['dt'].dt.hour
t['time_of_day'] = t['hour_int'].apply(time_bucket)

# cross-tab: each row is a shape, values are the % of that shape's sightings in each bucket
order = ['Morning (5AM-11AM)', 'Afternoon (12PM-4PM)', 'Evening (5PM-7PM)', 'Night (8PM-4AM)']
ct = pd.crosstab(t['shape_group'], t['time_of_day'], normalize='index')[order] * 100
display(ct.round(1))

fig = px.imshow(ct.round(1), text_auto=True, color_continuous_scale='Blues', aspect='auto',
                labels=dict(x='Time of day', y='Shape', color='% of shape'),
                title='Percent of Each Shape Seen by Time of Day')
fig.show()

This data strengthens our findings from graph 5. The greatest number of sightings occur during the dark hours of nighttime and early morning. Consequently, peak light hours (12pm-4pm) see the least amount.

# Areas of Improvement
Currently, our data sets cover broad areas of UFO sighting data, but we want to find data that is more recent, to help us create more relevant predictions. Additionally, we would like to find a dataset pertaining to the sightings of USOs to investigate the coastline element of our hypothesis. Regarding existing data, despite missing large amounts of data regarding the color observed in the second data set, we want to find a way to incorporate it into our nalaysis.